In [22]:
#Load android feature selected dataset for modeling:
import pandas as pd
android_selected_features = pd.read_csv("../Processed_Data/Selected_FeaturesDatasets/android_SelectedFeatures.csv")

**Train/Test Split**

In [23]:
#Preparing dataset for training:
#Select relevant columns for analysis:
y = android_selected_features['stress']
X = android_selected_features.drop(columns=['stress', 'uid', 'day'])  # Drop target, ID columns, and date
print("Final feature set columns:", X.columns)
print("Final feature set shape:", X.shape)  

Final feature set columns: Index(['Unnamed: 0', 'race_alaskan native/white', 'audio_amp_mean_ep_2',
       'act_in_vehicle_ep_0', 'race_american indian/alaska native',
       'light_mean_ep_3', 'race_american indian/white', 'sse3-4',
       'audio_amp_std_ep_2', 'pam', 'race_asian', 'race_black',
       'act_still_ep_3', 'race_more than one', 'phq4-1', 'phq4-2', 'gender',
       'phq4_score', 'race_other/hispanic', 'phq4-4', 'loc_self_dorm_dur',
       'sse3-1', 'sse3-3', 'race_white'],
      dtype='str')
Final feature set shape: (7256, 24)


In [3]:
#Splitting data into test and train sets to prevent data leakage:
from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold

#Column that identifies groups (participants):
group_col = 'uid'

#80/20 training testing split, with one testing group:
gss = GroupShuffleSplit(test_size=0.2, n_splits=1, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=android_selected_features[group_col]))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
groups_train = android_selected_features[group_col].iloc[train_idx]

#Checking stress label distribution to ensure the groups are stratified:
print("Train distribution:")
print(y_train.value_counts(normalize=True))

print("\nTest distribution:")
print(y_test.value_counts(normalize=True))

Train distribution:
stress
2.0    0.345873
3.0    0.253521
1.0    0.253165
4.0    0.110537
5.0    0.036905
Name: proportion, dtype: float64

Test distribution:
stress
3.0    0.295689
2.0    0.282332
1.0    0.239830
4.0    0.100182
5.0    0.081967
Name: proportion, dtype: float64


In [4]:
#Stratified group k-fold cross-validation to evaluate model performance:
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
for fold, (train_idx, val_idx) in enumerate(sgkf.split(X_train, y_train, groups=groups_train)):
    X_fold_train, X_fold_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

**Random Forest Regressor - Global**

In [5]:
#Random forest regression model with stratifed group k-fold cross-validation:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score    


rf_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=10,
    min_samples_split=10,
    min_samples_leaf=5,
    max_features='sqrt', 
    random_state=42,
    n_jobs=-1)
RF_fold_mse = []
RF_fold_r2 = []

for fold, (train_idx, val_idx) in enumerate(sgkf.split(X_train, y_train, groups=groups_train)):
    X_fold_train, X_fold_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
    
    rf_model.fit(X_fold_train, y_fold_train)
    val_preds = rf_model.predict(X_fold_val)
    
    mse = mean_squared_error(y_fold_val, val_preds)
    r2 = r2_score(y_fold_val, val_preds)
    RF_fold_mse.append(mse)
    RF_fold_r2.append(r2)
    print(f"Fold {fold+1} - MSE: {mse:.4f}, R^2: {r2:.4f}")

    
print(f"\nAverage MSE across folds: {sum(RF_fold_mse)/len(RF_fold_mse):.4f}")
print(f"Average R^2 across folds: {sum(RF_fold_r2)/len(RF_fold_r2):.4f}") 

Fold 1 - MSE: 0.8910, R^2: 0.3549
Fold 2 - MSE: 0.8308, R^2: 0.2356
Fold 3 - MSE: 0.6599, R^2: 0.4499
Fold 4 - MSE: 0.5235, R^2: 0.4781
Fold 5 - MSE: 0.7084, R^2: 0.3842

Average MSE across folds: 0.7227
Average R^2 across folds: 0.3806


In [6]:
#Final evaluation on the test set:
rf_model.fit(X_train, y_train)
test_preds = rf_model.predict(X_test)
test_mse = mean_squared_error(y_test, test_preds)
test_r2 = r2_score(y_test, test_preds)
print(f"\nTest Set - MSE: {test_mse:.4f}, R^2: {test_r2:.4f}")


Test Set - MSE: 1.0408, R^2: 0.2680


In [7]:
#Functions to convert regression predictions to class labels and compute accuracy:
from sklearn.metrics import accuracy_score
import numpy as np

# Define function to convert regression predictions to class labels based on binning:
def regression_to_class(y_pred):
    # Define bin edges
    bins = [1.5, 2.5, 3.5, 4.5]
    
    # Convert to class labels 1–5
    y_class = np.digitize(y_pred, bins) + 1
    
    return y_class

# Define function to compute per-class accuracy:
def per_class_accuracy(y_true, y_pred_class):
    classes = np.unique(y_true)
    class_acc = {}

    for c in classes:
        idx = (y_true == c)
        if np.sum(idx) == 0:
            class_acc[c] = np.nan
        else:
            class_acc[c] = np.mean(y_pred_class[idx] == c)

    return class_acc

In [8]:
#Computing accuracy for by converting regression predictions to class labels:
# Regression predictions
y_pred_reg = rf_model.predict(X_test)

# Convert to classes
y_pred_reg = np.clip(y_pred_reg, 1, 5)
y_pred_class = regression_to_class(y_pred_reg)
y_test_int = y_test.astype(int)

# Accuracy
acc = accuracy_score(y_test_int, y_pred_class)

# Compute per-class accuracy
class_acc = per_class_accuracy(y_test_int, y_pred_class)

print(f"Regression → Classification Accuracy: {acc:.2f}")
print("\nPer-class accuracy:")
for c, acc in class_acc.items():
    print(f"Class {c}: {acc:.3f}")


Regression → Classification Accuracy: 0.37

Per-class accuracy:
Class 1: 0.215
Class 2: 0.647
Class 3: 0.433
Class 4: 0.109
Class 5: 0.000


**XGBoost Regressor - Global**

In [9]:
#XGBoost regression model with stratifed group k-fold cross-validation:
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score

xgb_model = XGBRegressor(
    n_estimators=200,
    max_depth=10,
    learning_rate=0.01,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1)

XGB_fold_mse = []
XGB_fold_r2 = []

for fold, (train_idx, val_idx) in enumerate(sgkf.split(X_train, y_train, groups=groups_train)):
    X_fold_train, X_fold_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
    
    xgb_model.fit(X_fold_train, y_fold_train)
    val_preds = xgb_model.predict(X_fold_val)
    
    mse = mean_squared_error(y_fold_val, val_preds)
    r2 = r2_score(y_fold_val, val_preds)
    XGB_fold_mse.append(mse)
    XGB_fold_r2.append(r2)
    print(f"Fold {fold+1} - MSE: {mse:.4f}, R^2: {r2:.4f}")

print(f"\nAverage MSE across folds: {sum(XGB_fold_mse)/len(XGB_fold_mse):.4f}")
print(f"Average R^2 across folds: {sum(XGB_fold_r2)/len(XGB_fold_r2):.4f}")

Fold 1 - MSE: 0.9600, R^2: 0.3050
Fold 2 - MSE: 0.8727, R^2: 0.1971
Fold 3 - MSE: 0.7089, R^2: 0.4091
Fold 4 - MSE: 0.5511, R^2: 0.4507
Fold 5 - MSE: 0.7251, R^2: 0.3697

Average MSE across folds: 0.7635
Average R^2 across folds: 0.3463


In [10]:
#Final evaluation on the test set:
xgb_model.fit(X_train, y_train)
test_preds = xgb_model.predict(X_test)
test_mse = mean_squared_error(y_test, test_preds)
test_r2 = r2_score(y_test, test_preds)
print(f"\nTest Set - MSE: {test_mse:.4f}, R^2: {test_r2:.4f}")


Test Set - MSE: 1.0895, R^2: 0.2337


In [11]:
#Computing accuracy for by converting regression predictions to class labels:
# Regression predictions
y_pred_reg = xgb_model.predict(X_test)

# Convert to classes
y_pred_reg = np.clip(y_pred_reg, 1, 5)
y_pred_class = regression_to_class(y_pred_reg)
y_test_int = y_test.astype(int)

# Accuracy
acc = accuracy_score(y_test_int, y_pred_class)

# Compute per-class accuracy
class_acc = per_class_accuracy(y_test_int, y_pred_class)

print(f"Regression → Classification Accuracy: {acc:.2f}")
print("\nPer-class accuracy:")
for c, acc in class_acc.items():
    print(f"Class {c}: {acc:.3f}")


Regression → Classification Accuracy: 0.36

Per-class accuracy:
Class 1: 0.144
Class 2: 0.701
Class 3: 0.402
Class 4: 0.079
Class 5: 0.000


**XGBoost Regressor - Personalized**

In [12]:
from sklearn.model_selection import train_test_split

#Training personalized models for each participant:
unique_participants = android_selected_features['uid'].unique()

P_XGB_models = {}
P_XGB_train_metrics = {}

#Loop through each participant and train a personalized model:
for participant in unique_participants:
    
    participant_data = android_selected_features[
        android_selected_features['uid'] == participant
    ].sort_values('day') # Ensure data is sorted by day for time-based splitting
    
    X = participant_data.drop(columns=['stress', 'uid', 'day'])
    y = participant_data['stress']
    
    # Skip participants with too few data points for training:
    if len(participant_data) < 15:
        continue
    
    #Split into train/validation/test (60/20/20) with time-based splitting:
    X_train_full, X_test, y_train_full, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )
    
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_full, y_train_full, test_size=0.25, random_state=42
    )
    
    P_XGB_model = XGBRegressor(
        n_estimators=500,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1
    )
    
    #Train with validation:
    P_XGB_model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=False,
    )
    
    train_preds = P_XGB_model.predict(X_train)
    val_preds = P_XGB_model.predict(X_val)
    
    train_mse = mean_squared_error(y_train, train_preds)
    val_mse = mean_squared_error(y_val, val_preds)
    
    P_XGB_train_metrics[participant] = {
        'train_mse': train_mse,
        'val_mse': val_mse
    }
    
    # Store model + test data for later
    P_XGB_models[participant] = {
        'model': P_XGB_model,
        'X_test': X_test,
        'y_test': y_test
    }

In [13]:
#Calculate and print average training and validation MSE across all personalized models:
avg_train_mse = sum(m['train_mse'] for m in P_XGB_train_metrics.values()) / len(P_XGB_train_metrics)
avg_val_mse = sum(m['val_mse'] for m in P_XGB_train_metrics.values()) / len(P_XGB_train_metrics)

print(f"Average TRAIN MSE: {avg_train_mse:.4f}")
print(f"Average VALIDATION MSE: {avg_val_mse:.4f}")

Average TRAIN MSE: 0.0005
Average VALIDATION MSE: 0.6581


In [14]:
#Calculate test metrics for each personalized model:
P_RF_test_metrics = {}

for participant, data in P_XGB_models.items():
    
    P_XGB_model = data['model']
    X_test = data['X_test']
    y_test = data['y_test']
    
    preds = P_XGB_model.predict(X_test)
    
    mse = mean_squared_error(y_test, preds)
    r2 = r2_score(y_test, preds)
    
    P_RF_test_metrics[participant] = {
        'test_mse': mse,
        'test_r2': r2
    }

In [15]:
#Calculate and print average test metrics across all personalized models:
avg_test_mse = sum(m['test_mse'] for m in P_RF_test_metrics.values()) / len(P_RF_test_metrics)
avg_test_r2 = sum(m['test_r2'] for m in P_RF_test_metrics.values()) / len(P_RF_test_metrics)

print(f"\nAverage TEST MSE (FINAL): {avg_test_mse:.4f}")
print(f"Average TEST R^2 (FINAL): {avg_test_r2:.4f}")


Average TEST MSE (FINAL): 0.6266
Average TEST R^2 (FINAL): 0.1816


In [16]:
#Accuracy of personalized models:
y_pred_reg = P_XGB_model.predict(X_test)

# Convert to classes
y_pred_reg = np.clip(y_pred_reg, 1, 5)
y_pred_class = regression_to_class(y_pred_reg)
y_test_int = y_test.astype(int)

# Accuracy
acc = accuracy_score(y_test_int, y_pred_class)

# Compute per-class accuracy
class_acc = per_class_accuracy(y_test_int, y_pred_class)

print(f"Regression → Classification Accuracy: {acc:.2f}")
print("\nPer-class accuracy:")
for c, acc in class_acc.items():
    print(f"Class {c}: {acc:.3f}")


Regression → Classification Accuracy: 0.45

Per-class accuracy:
Class 1: 0.500
Class 2: 0.667
Class 3: 0.000
Class 4: 0.000


**Random Forest Regressor - Personalized**

In [17]:
P_RF_models = {}
P_RF_train_metrics = {}

# Loop through each participant
for participant in unique_participants:
    
    participant_data = android_selected_features[
        android_selected_features['uid'] == participant
    ].sort_values('day')  # keep time ordering
    
    X = participant_data.drop(columns=['stress', 'uid', 'day'])
    y = participant_data['stress']
    
    # Skip small datasets
    if len(participant_data) < 15:
        continue
    
    # Split (same as your setup)
    X_train_full, X_test, y_train_full, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )
    
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_full, y_train_full, test_size=0.25, random_state=42
    )
    
    #Random Forest model
    P_RF_model = RandomForestRegressor(
        n_estimators=300,
        max_depth=None,
        min_samples_split=5,
        min_samples_leaf=2,
        max_features='sqrt',
        random_state=42,
        n_jobs=-1
    )
    
    # Train (no eval_set here)
    P_RF_model.fit(X_train, y_train)
    
    # Predictions
    train_preds = P_RF_model.predict(X_train)
    val_preds = P_RF_model.predict(X_val)
    
    # Metrics
    train_mse = mean_squared_error(y_train, train_preds)
    val_mse = mean_squared_error(y_val, val_preds)
    
    P_RF_train_metrics[participant] = {
        'train_mse': train_mse,
        'val_mse': val_mse
    }
    
    # Store model + test data
    P_RF_models[participant] = {
        'model': P_RF_model,
        'X_test': X_test,
        'y_test': y_test
    }

In [18]:
#Calculate and print average training and validation MSE across all personalized models:
avg_train_mse = sum(m['train_mse'] for m in P_RF_train_metrics.values()) / len(P_RF_train_metrics)
avg_val_mse = sum(m['val_mse'] for m in P_RF_train_metrics.values()) / len(P_RF_train_metrics)

print(f"Average TRAIN MSE: {avg_train_mse:.4f}")
print(f"Average VALIDATION MSE: {avg_val_mse:.4f}")

Average TRAIN MSE: 0.2359
Average VALIDATION MSE: 0.6156


In [19]:
#Calculate test metrics for each personalized model:
P_RF_test_metrics = {}

for participant, data in P_RF_models.items():
    
    P_RF_model = data['model']
    X_test = data['X_test']
    y_test = data['y_test']
    
    preds = P_RF_model.predict(X_test)
    
    mse = mean_squared_error(y_test, preds)
    r2 = r2_score(y_test, preds)
    
    P_RF_test_metrics[participant] = {
        'test_mse': mse,
        'test_r2': r2
    }

In [20]:
#Calculate and print average test metrics across all personalized models:
avg_test_mse = sum(m['test_mse'] for m in P_RF_test_metrics.values()) / len(P_RF_test_metrics)
avg_test_r2 = sum(m['test_r2'] for m in P_RF_test_metrics.values()) / len(P_RF_test_metrics)

print(f"\nAverage TEST MSE (FINAL): {avg_test_mse:.4f}")
print(f"Average TEST R^2 (FINAL): {avg_test_r2:.4f}")


Average TEST MSE (FINAL): 0.5512
Average TEST R^2 (FINAL): 0.3230


In [21]:
#Computing accuracy for by converting regression predictions to class labels:
# Regression predictions
y_pred_reg = P_RF_model.predict(X_test)

# Convert to classes
y_pred_reg = np.clip(y_pred_reg, 1, 5)
y_pred_class = regression_to_class(y_pred_reg)
y_test_int = y_test.astype(int)

# Accuracy
acc = accuracy_score(y_test_int, y_pred_class)

# Compute per-class accuracy
class_acc = per_class_accuracy(y_test_int, y_pred_class)

print(f"Regression → Classification Accuracy: {acc:.2f}")
print("\nPer-class accuracy:")
for c, acc in class_acc.items():
    print(f"Class {c}: {acc:.3f}")

Regression → Classification Accuracy: 0.45

Per-class accuracy:
Class 1: 0.000
Class 2: 0.667
Class 3: 1.000
Class 4: 0.000
